# 02. Build the full-text control set (for coding)

Pulls **every eligible control opinion** from the bulk file with a wide text window around the
sanctions ruling, so each one can be coded by reading the actual text (regex on full opinions
plateaued at held-out kappa 0.36).

**Outputs** (to `../data/coded/`):
- `controls_all_fulltext.csv` — every eligible control, with `text_for_coding` and a reference
  regex label. This is the file to code.
- `controls_validation_40.csv` — a 40-row random subset for independent hand-validation.

Eligible = 2023–2026, sanctions-triggered, not AI-contaminated, not bar-discipline, not criminal
community-control / licensing. Run from `code/`. Needs the bulk file `02` uses.

### Config

In [1]:
import re, sys, os
import pandas as pd, numpy as np
sys.path.append("labeling"); import label_lib as L

SNAPSHOT_DATE = "2026-06-30"
BULK_CSV = f"../data/raw/bulk-data/opinion-clusters-{SNAPSHOT_DATE}.csv.bz2"
YEAR_MIN, YEAR_MAX = 2023, 2026
N_SAMPLE, N_QUICK, SEED = 200, 40, 73   # fresh seed for held-out validation
CHUNKSIZE = 100_000

SANCTION_TRIGGER = re.compile(
    r"rule\s*11|§?\s*1927|section\s*1927|inherent\s+authority|sanction|"
    r"show\s+cause|disciplin|referr", re.I)
READ_KW = dict(engine="c", quotechar='"', escapechar="\\", on_bad_lines="skip", low_memory=False, dtype=str)
COL = dict(case="case_name", date="date_filed", nos="nature_of_suit", attorneys="attorneys",
           text_cols=["disposition","summary","procedural_history","posture","syllabus","headnotes"])
KEEP = set([COL["case"],COL["date"],COL["nos"],COL["attorneys"]]+COL["text_cols"])

### Helpers

In [2]:
def build_text(df):
    cols=[c for c in COL["text_cols"] if c in df.columns]
    s=pd.Series("", index=df.index, dtype="object")
    for c in cols:
        s=s.str.cat(df[c].fillna("").astype(str), sep="  ")
    return s

def wide_region(text, before=400, after=2200):
    """A generous window around the sanctions trigger, so a human can read the ruling."""
    if not isinstance(text,str) or not text: return ""
    m=SANCTION_TRIGGER.search(text)
    if not m: return text[:1500]
    i=m.start(); return text[max(0,i-before): i+after]

def eligible(df):
    """Same population as the controls: window + sanctions trigger, minus AI-contaminated and
    bar-discipline. Returns rows with a full text blob to sample from."""
    df=df.rename(columns={COL["case"]:"case_name", COL["date"]:"date_filed", COL["nos"]:"nature_of_suit"})
    df["date"]=pd.to_datetime(df.get("date_filed"), errors="coerce"); df["year"]=df["date"].dt.year
    df=df[df["year"].between(YEAR_MIN,YEAR_MAX)]
    if len(df)==0: return pd.DataFrame(columns=["case_name","date_filed","year","nature_of_suit","_text"])
    df["_text"]=build_text(df)
    keep=(df["_text"].str.contains(SANCTION_TRIGGER,na=False)
          & ~df["_text"].map(L.is_ai_contaminated)
          & ~df["_text"].map(L.is_bar_discipline)
          & ~df["_text"].map(L.is_criminal_or_licensing))
    df=df[keep]
    return df[["case_name","date_filed","year","nature_of_suit","_text"]]

### Stream the bulk file and collect eligible cases

In [3]:
assert os.path.exists(BULK_CSV), f"Not found: {BULK_CSV}"
reader=pd.read_csv(BULK_CSV, usecols=lambda c: c in KEEP, chunksize=CHUNKSIZE, **READ_KW)
pool=[]; scanned=0
for i,ch in enumerate(reader):
    pool.append(eligible(ch)); scanned+=len(ch)
    if (i+1)%10==0: print(f"  scanned {scanned:>10,} | eligible so far {sum(len(p) for p in pool):>5,}")
elig=pd.concat(pool, ignore_index=True) if pool else pd.DataFrame()
print(f"\ntotal eligible controls: {len(elig):,}")

  scanned  1,000,000 | eligible so far   219
  scanned  2,000,000 | eligible so far   387
  scanned  3,000,000 | eligible so far   488
  scanned  4,000,000 | eligible so far   527
  scanned  5,000,000 | eligible so far   542
  scanned  6,000,000 | eligible so far   559
  scanned  7,000,000 | eligible so far   588
  scanned  8,000,000 | eligible so far   690
  scanned  9,000,000 | eligible so far   748
  scanned 10,000,000 | eligible so far   793

total eligible controls: 797


### Sample, label with the coder, write the validation files

In [4]:
# keep ALL eligible controls (with full text) so every one can be coded in chat
allc = elig.reset_index(drop=True).copy()
allc["coder_severity"] = allc["_text"].map(lambda t: L.code_severity(t, is_full_opinion=True))  # regex label, for reference
allc["text_for_coding"] = allc["_text"].map(wide_region)
allc["hand_severity"] = ""          # <- filled by the coder (Claude in chat, or you)
out_cols = ["case_name","date_filed","year","nature_of_suit","coder_severity","hand_severity","text_for_coding"]

# full set -> this is the file to upload for coding
allc[out_cols].to_csv("../data/coded/controls_all_fulltext.csv", index=False)

# a 40-row random subset -> YOUR independent hand-validation sheet
allc[out_cols].sample(min(N_QUICK, len(allc)), random_state=SEED).to_csv(
    "../data/coded/controls_validation_40.csv", index=False)

print(f"wrote ../data/coded/controls_all_fulltext.csv  ({len(allc)} controls)")
print("wrote ../data/coded/controls_validation_40.csv  (40-row validation subset)")
print("regex coder_severity dist (reference):", allc["coder_severity"].value_counts().sort_index().to_dict())

wrote ../data/coded/controls_all_fulltext.csv  (797 controls)
wrote ../data/coded/controls_validation_40.csv  (40-row validation subset)
regex coder_severity dist (reference): {0: 569, 1: 3, 2: 77, 3: 65, 4: 83}
